In [ ]:
# Cell 1: TF device check
import sys, tensorflow as tf, os
print("Python executable:", sys.executable)
print("TF version:", tf.__version__)
print("Available devices:")
for d in tf.config.list_physical_devices():
    print(d)

In [ ]:
# Cell 2: clone repo (adjust baseDir & branch)
import os, shutil
baseDir = os.path.expanduser('~/Documents/NeuroSpeech')  # change to your path
branch = "Shreyaj-Padigala" #change to your name
repo_url = "https://github.com/Aditya-Yan/NeuroSpeech.git"

if not os.path.exists(baseDir):
    print("Cloning repo...")
    !git clone --branch {branch} --single-branch {repo_url} "{baseDir}"
else:
    print("Repo exists; fetching latest for branch", branch)
    os.chdir(baseDir)
    !git fetch origin {branch}
    !git checkout {branch}
    !git pull origin {branch}

os.makedirs(os.path.join(baseDir,'speechBCI-main','derived','rnns'), exist_ok=True)

In [ ]:
# Cell 3: run training (HM-RNN)
nUnits = [256,256,256]   # list -> units per hierarchical layer
kernel_size = 16
time_scales = [1,4,16]   # informational for config
batch_size = 32
nBatchesToTrain = 100
learnRateStart = 0.02
learnRateDecaySteps = 10000

base_run_dir = os.path.join(baseDir, 'speechBCI-main', 'derived', 'rnns', 'hmrnnTrial')

# Run via hydra main with model.type=hmrnn
# Use python -m invocation to avoid notebook %run reimport issues
!python -m neuralDecoder.main \
    dataset=speech_release_baseline \
    model.type=hmrnn \
    model.nUnits={nUnits} \
    model.stack_kwargs.kernel_size={kernel_size} \
    model.time_scales={time_scales} \
    learnRateStart={learnRateStart} \
    learnRateDecaySteps={learnRateDecaySteps} \
    nBatchesToTrain={nBatchesToTrain} \
    --hydra.run.dir={base_run_dir}

In [ ]:
# Cell 4: visualize
import scipy.io
import matplotlib.pyplot as plt
import numpy as np

snapshot_path = os.path.join(base_run_dir, 'outputSnapshot.mat')
print("Loading", snapshot_path)
dat = scipy.io.loadmat(snapshot_path)
print(dat.keys())

if 'logitsSnapshot' in dat:
    plt.figure(figsize=(10,4))
    plt.imshow(dat['logitsSnapshot'].T, aspect='auto')
    plt.title('Logits snapshot')
    plt.show()

if 'inputFeaturesSnapshot' in dat:
    plt.figure(figsize=(10,4))
    plt.imshow(dat['inputFeaturesSnapshot'].T, aspect='auto')
    plt.title('Input features snapshot')
    plt.show()

if 'z_prob' in dat:
    plt.figure(figsize=(10,4))
    plt.imshow(dat['z_prob'].T, aspect='auto')
    plt.title('HMRNN boundary probabilities (z_prob) snapshot')
    plt.colorbar()
    plt.show()
